# 4 · Live demo — answer-level score + per-sentence hallucination localization

Scores questions end-to-end (answer + 3 detector scores + fused probability) and **highlights which sentence** of a longer answer is likely hallucinated.

**Single model (fast, interactive).** All three detectors run on ONE Instruct model (~6 GB, fits 12 GB easily) — load it once, then every `pipe.score(q)` takes a couple of seconds. This needs the **Instruct-trained TSV** from notebook 1b; if you haven't run 1b, run it first (otherwise TSV is on the base model and you'd need the slow two-model path).

*(Per-sentence scores are indicative; the calibrated number is the answer-level fused score.)*

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
import torch, gc
from pipeline import HallKingPipeline
from fusion import FusionModel
from localize import localize, render_highlight
DATASET='triviaqa'
gc.collect(); torch.cuda.empty_cache()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM before load: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total')
# separate_tsv=False -> ONE Instruct model for SEP+HalluShift+TSV (TSV re-trained in nb 1b).
pipe = HallKingPipeline(dataset=DATASET, separate_tsv=False, retrained=True).load()
pipe.fusion = FusionModel.load(os.path.join('..','models',f'fusion_{DATASET}_oof.pkl'))
print('pipeline + fusion ready (single model)')

### Answer-level scoring (interactive — each call is ~seconds, no reload)

In [ ]:
for q in ['What is the capital of Australia?',
          'Who invented the telephone?',
          'What is the largest planet in the solar system?']:
    r = pipe.score(q)
    print(f"Q: {q}\n  A: {r['answer']!r}")
    print(f"  sep_entropy={r['sep_entropy']:.2f} hallushift={r['hallushift']:.2f} "
          f"tsv_margin={r['tsv_margin']:.3f}  ->  FUSED P(halluc)={r['fused']:.2f}\n")

### Score your own question
Change `q` and re-run — the model stays loaded, so it's instant.

In [ ]:
q = 'What year did the first human land on Mars?'
r = pipe.score(q)
print('A:', repr(r['answer']))
print(f"sep_entropy={r['sep_entropy']:.2f} hallushift={r['hallushift']:.2f} "
      f"tsv_margin={r['tsv_margin']:.3f}  ->  FUSED P(halluc)={r['fused']:.2f}")

### Per-sentence localization on a longer answer
`use_claim_filter=True` skips filler/meta sentences (loads a small DeBERTa NLI judge once); set it to `False` to score every sentence without the extra model.

In [ ]:
q = 'Tell me three facts about the planet Mars.'
res = localize(pipe, q, max_new_tokens=200, use_claim_filter=True)
print('ANSWER:\n', res['answer'], '\n')
print('PER-SENTENCE (tier · fused probability; fillers shown as filler/None):')
for s in res['sentences']:
    f = 'filler' if s['fused'] is None else f"{s['fused']:.2f}"
    print(f"  [{s['tier']:6s} {f:>6}] {s['sentence']}")